# DuckSeg: protein recruitment to laser-induced DNA damage

<a href="https://colab.research.google.com/github/ronalabrcns/DuckSeg/blob/main/notebooks/DuckSeg_Example_Form.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

This notebook runs the full [DuckSeg](https://github.com/ronalabrcns/DuckSeg) pipeline **without editing any code**. Every step below is a form: fill in the fields on the right, then press the ▶ button on the left of the step. Run the steps from top to bottom. To see the code behind a step, double-click it.

**Before you start**

* *Runtime → Change runtime type → GPU* (T4 is enough).
* **Accept the CellSAM model license (once).** DuckSeg is MIT-licensed, but the CellSAM model weights are third-party and licensed for **non-commercial academic use only**. Log in at [huggingface.co](https://huggingface.co) (a free account is enough), open [vanvalenlab/cellsam](https://huggingface.co/vanvalenlab/cellsam) and click *I agree to the license terms*.
* **Hugging Face token.** Create a *Read* token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). In Colab click the 🔑 *Secrets* icon on the left, add a secret named `HF_TOKEN` with the token as value, and switch on *Notebook access*.
  *(A DeepCell token from [users.deepcell.org](https://users.deepcell.org/), saved as `DEEPCELL_ACCESS_TOKEN`, still works instead.)*

Documentation: [duckseg.readthedocs.io](https://duckseg.readthedocs.io/en/latest/workflow.html) · The same pipeline as editable code: [DuckSeg_Example.ipynb](https://colab.research.google.com/github/ronalabrcns/DuckSeg/blob/main/notebooks/DuckSeg_Example.ipynb)

In [ ]:
#@title 1. Install DuckSeg and CellSAM (2-3 minutes)
#@markdown Installs everything and then restarts the runtime once. Colab reports *"Your session crashed"* or reconnects: **that is expected**. Then continue with step 2 (running this step again afterwards does nothing).
import importlib.util, os

if importlib.util.find_spec("DuckSeg") and importlib.util.find_spec("cellSAM"):
    print("DuckSeg and CellSAM are already installed. Continue with step 2.")
else:
    !git clone --quiet --depth 1 https://github.com/ronalabrcns/DuckSeg.git 2>/dev/null || true
    !pip install --quiet -e ./DuckSeg
    !pip install --quiet git+https://github.com/vanvalenlab/cellSAM.git@362a8e521bcd7313d5cb280bcd4cfc12977c1e9f
    !pip install --quiet ipympl
    print("Installed. Restarting the runtime ...")
    os.kill(os.getpid(), 9)

In [ ]:
#@title 2. Accept the license and load the CellSAM model
#@markdown The CellSAM weights are downloaded once from [Hugging Face](https://huggingface.co/vanvalenlab/cellsam). They are licensed under a [modified Apache 2.0 license](https://huggingface.co/vanvalenlab/cellsam/blob/main/LICENSE) for **non-commercial academic use only**, separate from DuckSeg's MIT license.
i_accept_the_cellsam_license = False #@param {type:"boolean"}
#@markdown Your Hugging Face token is read from the Colab *Secrets* panel (`HF_TOKEN`), never typed in here, so it is not saved in the notebook.
import os, sys

if not i_accept_the_cellsam_license:
    raise SystemExit("Tick 'i_accept_the_cellsam_license' above (and accept the license on "
                     "huggingface.co/vanvalenlab/cellsam), then run this step again.")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    sys.path.insert(0, "/content/DuckSeg/src")   # in case the editable install is not picked up yet
    from google.colab import output, userdata
    output.enable_custom_widget_manager()        # needed for the interactive widgets
    for secret in ("HF_TOKEN", "DEEPCELL_ACCESS_TOKEN"):
        try:
            os.environ[secret] = userdata.get(secret)
        except Exception:
            pass
    if not (os.environ.get("HF_TOKEN") or os.environ.get("DEEPCELL_ACCESS_TOKEN")):
        raise SystemExit("No token found. Add your Hugging Face token to Colab Secrets as HF_TOKEN "
                         "and switch on notebook access (see 'Before you start').")

import torch
from DuckSeg.model_loader import load_cellsam_model
import DuckSeg.experiment_evaluator as evaluator

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow!)")
if evaluator.cellsam_model is None:
    evaluator.cellsam_model = load_cellsam_model()   # reused by every segmentation step below
print("CellSAM model loaded")

In [ ]:
#@title 3. Choose the data
#@markdown **example** downloads the example dataset. **drive** uses your own recordings from Google Drive: one folder per condition, with one video and one laser-ROI mask per experiment, e.g. `MSH2_001.nd2` + `MSH2_001_ROI.tif` ([Prepare your data](https://duckseg.readthedocs.io/en/latest/guides/data_preparation.html)).
data_source = "example" #@param ["example", "drive"]
#@markdown ---
#@markdown **Only for `drive`:**
drive_folder = "/content/drive/MyDrive/DuckSeg/my_condition" #@param {type:"string"}
condition_name = "my_condition" #@param {type:"string"}
#@markdown ---
#@markdown Frames to analyse per video (`0` = the length of the shortest video, so that all videos share one time axis):
max_frames = 0 #@param {type:"integer"}
show_stripe_check = True #@param {type:"boolean"}
#@markdown The stripe check blacks out the detected laser stripe on the first two videos. It should sit on the irradiated line; if it does not, use step 3b.
import glob, urllib.request, zipfile
from DuckSeg.experiment_loader import ExperimentBatch

EXAMPLE_DATA_URL = ""      # TODO: Zenodo / GitHub release link of the example dataset
EXAMPLE_NAME = "MSH2"

if data_source == "example":
    if not EXAMPLE_DATA_URL:
        raise SystemExit("The example dataset is not online yet. Choose data_source = drive and use your own data.")
    data_root = "/content/example_data" if IN_COLAB else os.path.abspath("example_data")
    if not os.path.isdir(data_root):
        os.makedirs(data_root)
        print("Downloading example data ...")
        zip_path, _ = urllib.request.urlretrieve(EXAMPLE_DATA_URL, os.path.join(data_root, "data.zip"))
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_root)
        os.remove(zip_path)
    data_dir = os.path.dirname(sorted(glob.glob(os.path.join(data_root, "**", "*_ROI.tif"), recursive=True))[0])
    name = EXAMPLE_NAME
else:
    if IN_COLAB and drive_folder.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.isdir(drive_folder):
        raise SystemExit(f"Folder not found: {drive_folder}")
    data_dir, name = drive_folder, condition_name.strip() or "my_condition"

batch = ExperimentBatch(name, data_dir)
experiments = batch.experiments()
if not experiments:
    raise SystemExit(f"No videos with a matching *_ROI.tif mask found in {data_dir}")
shortest = min(e.frame_count() for e in experiments)
batch.max_len = min(max_frames, shortest) if max_frames > 0 else shortest
print(f"{len(experiments)} experiments in {data_dir}:", [e.name for e in experiments])
print("Frames per video:", batch.max_len)

if show_stripe_check:
    for experiment in experiments[:2]:
        experiment.plot_frame_with_roi_masked()

In [ ]:
#@title 3b. Optional: move the laser stripe
#@markdown Tick the box and run this step to get an editor per video: move the stripe with the ▲ / ▼ buttons. Changes are saved automatically.
edit_roi = False #@param {type:"boolean"}
if edit_roi:
    %matplotlib widget
    from DuckSeg.roi_editor import show_roi_editors
    show_roi_editors(batch.experiments())
else:
    print("Skipped (tick edit_roi to open the editor).")

In [ ]:
#@title 4. Tune the preprocessing (quick preview)
#@markdown These steps are applied to every frame before CellSAM. Change the values and rerun this step until most nuclei are outlined in the preview ([Tune the preprocessing](https://duckseg.readthedocs.io/en/latest/guides/preprocessing.html)).
brightness = 20 #@param {type:"slider", min:1, max:100, step:1}
#@markdown Scales every frame to the same mean brightness.
contrast = 1.0 #@param {type:"slider", min:0.5, max:3.0, step:0.1}
#@markdown `1.0` = unchanged.
use_threshold = True #@param {type:"boolean"}
threshold = 20 #@param {type:"slider", min:0, max:255, step:1}
#@markdown Turns the frame into a black-and-white image (a clean, strong signal for the nuclei).
#@markdown ---
#@markdown **Preview**
preview_experiment = 0 #@param {type:"integer"}
preview_frames = 2 #@param {type:"slider", min:1, max:6, step:1}
#@markdown Number of frames spread over the video (2 = first and last).
show_intermediate_steps = False #@param {type:"boolean"}
%matplotlib inline
from DuckSeg.image_transforms import convert_to_u8, set_brightness, threshold_image, adjust_contrast
from DuckSeg.image_transformer import generate_masks_using_filters_and_transforms

transforms = [convert_to_u8, set_brightness(brightness)]   # CellSAM needs 8-bit images
if contrast != 1.0:
    transforms.append(adjust_contrast(alpha=contrast))
if use_threshold:
    transforms.append(threshold_image(threshold))

generate_masks_using_filters_and_transforms(
    batch,
    experiment_filter=[preview_experiment],
    frame_filter=preview_frames,
    transforms=transforms,
    show_after_transform=list(range(len(transforms))) if show_intermediate_steps else [],
)

In [ ]:
#@title 5. Segment and track all cells (slow step)
#@markdown CellSAM segments every frame and each cell touching the stripe is followed through the video. This takes a few minutes on a GPU. Results are cached next to the videos, so running it again is instant.
#@markdown **Untick `reuse_cache` after you changed the preprocessing in step 4**, otherwise the old masks are used.
reuse_cache = True #@param {type:"boolean"}
from DuckSeg.experiment_evaluator import evaluate_batch

vids = evaluate_batch(batch, transforms=transforms, use_cache=reuse_cache)
print(f"{len(vids)} tracked cells")

In [ ]:
#@title 6. Filter the cells automatically
#@markdown Removes cells whose masks look implausible ([Review and filter cells](https://duckseg.readthedocs.io/en/latest/guides/outlier_filtering.html)).
range_modifier = 4 #@param {type:"slider", min:1, max:10, step:1}
#@markdown Width of the scored band, as a multiple of the drawn stripe.
#@markdown ---
min_area = 10 #@param {type:"integer"}
max_area = 5000 #@param {type:"integer"}
#@markdown Allowed mask area in pixels, in every frame.
limit_area_jumps = True #@param {type:"boolean"}
max_area_change = 0.3 #@param {type:"slider", min:0.05, max:1.0, step:0.05}
#@markdown Largest allowed change of the mask area between two frames (0.3 = 30 %).
always_exclude = "" #@param {type:"string"}
#@markdown Cell ids to always exclude, separated by commas, e.g. `MSH2_001_3, MSH2_002_7`.
from DuckSeg.outlier_filters import outlier_vids, area_between, max_area_change_between_frames

filters = [area_between(min_area, max_area)]
if limit_area_jumps:
    filters.append(max_area_change_between_frames(max_area_change))
extra_manual_outliers = {s.strip() for s in always_exclude.split(",") if s.strip()}

outliers = outlier_vids(vids, extra_manual_outliers, filters)
print(f"{len(outliers)} of {len(vids)} cells removed by the filters")

In [ ]:
#@title 7. Review the cells by eye (optional)
#@markdown Shows a video of every cell. **Tick a cell's checkbox to exclude it**; a red border means it is already excluded by step 6. Your choices are used by steps 8 and 9 (rerunning this step clears them).
show_review_grid = True #@param {type:"boolean"}
cells_per_grid = 30 #@param {type:"slider", min:6, max:60, step:6}
from DuckSeg.cell_video_visualizer import normalize_videos_for_display, show_video_grid_jupyter

manual_outliers = set()            # filled by the checkboxes below
if show_review_grid:
    display_vids = normalize_videos_for_display(vids, range_modifier=range_modifier)
    for i in range(0, len(display_vids), cells_per_grid):
        update = show_video_grid_jupyter(display_vids[i:i + cells_per_grid], manual_outliers=manual_outliers)
        update(outliers, extra_manual_outliers)
else:
    print("Skipped.")

In [ ]:
#@title 8. Plot the recruitment curve
#@markdown Mean over the kept cells, with the standard error of the mean as a band. **Ratio** = inside / outside the band, **Diff** = inside − outside ([Concepts](https://duckseg.readthedocs.io/en/latest/concepts.html)).
score = "Ratio and Diff" #@param ["Ratio and Diff", "Ratio", "Diff"]
baseline_correct = True #@param {type:"boolean"}
#@markdown Diff curves start at 0 and Ratio curves at 1.
plot_title = "" #@param {type:"string"}
show_3d_traces = False #@param {type:"boolean"}
#@markdown The 3D plot shows every cell's trace, sorted by its mean, so unusual cells stand out.
from DuckSeg.experiment_evaluator import (ScoreType, plot_brightness_score,
                                          calculate_brightness_score_for_videos, plot_all_vid_scores)

manual_outliers = globals().get("manual_outliers", set())
cell_filter = lambda vid: vid.id not in outliers and vid.id not in manual_outliers
kept = [vid for vid in vids if cell_filter(vid)]
print(f"Analysing {len(kept)} of {len(vids)} cells")

os.makedirs("output", exist_ok=True)
score_types = {"Ratio and Diff": [ScoreType.Ratio, ScoreType.Diff],
               "Ratio": [ScoreType.Ratio], "Diff": [ScoreType.Diff]}[score]
for score_type in score_types:
    plot_brightness_score(batch, score_type=score_type, range_modifier=range_modifier,
                          cell_filter=cell_filter, baseline_correct=baseline_correct,
                          save_path=f"output/recruitment_{score_type.name.lower()}.png",
                          title=plot_title or None)

if show_3d_traces:
    ids, scores = calculate_brightness_score_for_videos(vids, score_type=score_types[0],
                                                        range_modifier=range_modifier, cell_filter=cell_filter)
    plot_all_vid_scores(scores, ids)

In [ ]:
#@title 9. Export the results
#@markdown Saves a CSV with one row per frame and four columns per kept cell: raw `_normal` and `_ROI` intensities and the `_diff` and `_ratio` scores ([Export results](https://duckseg.readthedocs.io/en/latest/guides/export_results.html)). The plots from step 8 are in the same `output` folder.
download_zip = True #@param {type:"boolean"}
#@markdown Download the CSV and the plots as one zip file.
from DuckSeg.video_exporter import export_cell_videos

csv_path = f"output/{name}_brightness_scores.csv"
export_cell_videos(kept, csv_path, range_modifier=range_modifier, baseline_correct=baseline_correct)
print("Saved", csv_path)

if download_zip and IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("duckseg_results", "zip", "output")
    files.download("duckseg_results.zip")

## Next steps

* Analyse another condition: change the folder in step 3 and run steps 3–9 again.
* Compare several conditions in one plot: [Compare conditions](https://duckseg.readthedocs.io/en/latest/guides/compare_conditions.html) (code notebook).
* Problems? See [Troubleshooting](https://duckseg.readthedocs.io/en/latest/troubleshooting.html).